In [2]:
import pandas as pd
import sys
from pathlib import Path

# Add project root to path so we can import config
sys.path.append(str(Path().resolve().parent))
import config

# Sanity check
print("RAW_DIR:", config.RAW_DIR)
print("File exists:", config.VARIANT_SUMMARY.exists())

RAW_DIR: D:\projects\claude_code\dissertation\clinvar\data\raw
File exists: True


In [3]:
# Peek at structure — first 5 rows only
df_peek = pd.read_csv(
    config.VARIANT_SUMMARY,
    sep='\t',
    nrows=5,
    low_memory=False
)

print("Shape:", df_peek.shape)
print("\nColumns:")
for col in df_peek.columns:
    print(" ", col)

Shape: (5, 43)

Columns:
  #AlleleID
  Type
  Name
  GeneID
  GeneSymbol
  HGNC_ID
  ClinicalSignificance
  ClinSigSimple
  LastEvaluated
  RS# (dbSNP)
  nsv/esv (dbVar)
  RCVaccession
  PhenotypeIDS
  PhenotypeList
  Origin
  OriginSimple
  Assembly
  ChromosomeAccession
  Chromosome
  Start
  Stop
  ReferenceAllele
  AlternateAllele
  Cytogenetic
  ReviewStatus
  NumberSubmitters
  Guidelines
  TestedInGTR
  OtherIDs
  SubmitterCategories
  VariationID
  PositionVCF
  ReferenceAlleleVCF
  AlternateAlleleVCF
  SomaticClinicalImpact
  SomaticClinicalImpactLastEvaluated
  ReviewStatusClinicalImpact
  Oncogenicity
  OncogenicityLastEvaluated
  ReviewStatusOncogenicity
  SCVsForAggregateGermlineClassification
  SCVsForAggregateSomaticClinicalImpact
  SCVsForAggregateOncogenicityClassification


In [4]:
# Load 10,000 rows to understand value distributions
df_sample = pd.read_csv(
    config.VARIANT_SUMMARY,
    sep='\t',
    nrows=10000,
    low_memory=False
)

# Key column distributions
print("=== ClinicalSignificance ===")
print(df_sample['ClinicalSignificance'].value_counts())

print("\n=== ReviewStatus ===")
print(df_sample['ReviewStatus'].value_counts())

print("\n=== Type ===")
print(df_sample['Type'].value_counts())

print("\n=== Assembly ===")
print(df_sample['Assembly'].value_counts())

print("\n=== Origin ===")
print(df_sample['Origin'].value_counts())

=== ClinicalSignificance ===
ClinicalSignificance
Pathogenic                                                                5731
Pathogenic/Likely pathogenic                                              1782
Conflicting classifications of pathogenicity                               720
Likely pathogenic                                                          640
Uncertain significance                                                     508
Benign                                                                     155
risk factor                                                                102
Benign/Likely benign                                                        90
Affects                                                                     37
no classification for the single variant                                    33
Likely benign                                                               32
association                                                                 32
dr

In [6]:
import gzip

# ── Filter settings ────────────────────────────────────────────────────────
KEEP_REVIEW = {
    "practice guideline",
    "reviewed by expert panel",
    "criteria provided, multiple submitters, no conflicts",
}

LABEL_MAP = {
    "Pathogenic":                      1,
    "Likely pathogenic":               1,
    "Pathogenic/Likely pathogenic":    1,
    "Benign":                          0,
    "Likely benign":                   0,
    "Benign/Likely benign":            0,
}

KEEP_ASSEMBLY = "GRCh38"

# ── Process in chunks ─────────────────────────────────────────────────────
chunks = []
chunk_size = 50_000
total_rows = 0
kept_rows = 0

reader = pd.read_csv(
    config.VARIANT_SUMMARY,
    sep='\t',
    chunksize=chunk_size,
    low_memory=False
)

for chunk in reader:
    total_rows += len(chunk)

    # 1. GRCh38 only
    chunk = chunk[chunk['Assembly'] == KEEP_ASSEMBLY]

    # 2. Germline only
    chunk = chunk[chunk['Origin'].str.contains('germline', na=False)]

    # 3. Review status filter
    chunk = chunk[chunk['ReviewStatus'].isin(KEEP_REVIEW)]

    # 4. Clean labels only
    chunk = chunk[chunk['ClinicalSignificance'].isin(LABEL_MAP.keys())]

    # 5. Map to binary label
    chunk['Label'] = chunk['ClinicalSignificance'].map(LABEL_MAP)

    kept_rows += len(chunk)
    chunks.append(chunk)

df_filtered = pd.concat(chunks, ignore_index=True)

print(f"Total rows processed: {total_rows:,}")
print(f"Rows after filtering: {kept_rows:,}")
print(f"Retention rate: {kept_rows/total_rows*100:.1f}%")
print(f"\nLabel distribution:")
print(df_filtered['Label'].value_counts())
print(f"\nClass ratio: {df_filtered['Label'].value_counts()[0] / df_filtered['Label'].value_counts()[1]:.1f}:1")

Total rows processed: 8,985,214
Rows after filtering: 383,533
Retention rate: 4.3%

Label distribution:
Label
0    288464
1     95069
Name: count, dtype: int64

Class ratio: 3.0:1


In [7]:
# Save filtered dataset
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_path = config.PROCESSED_DIR / "clinvar_filtered.tsv.gz"

df_filtered.to_csv(output_path, sep='\t', index=False, compression='gzip')
print(f"Saved to {output_path}")
print(f"Columns retained: {len(df_filtered.columns)}")
print(f"\nSample of key columns:")
df_filtered[['GeneSymbol', 'Type', 'ClinicalSignificance', 'Label', 
             'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'ReviewStatus']].head(10)

Saved to D:\projects\claude_code\dissertation\clinvar\data\processed\clinvar_filtered.tsv.gz
Columns retained: 44

Sample of key columns:


,GeneSymbol,Type,ClinicalSignificance,Label,ReferenceAlleleVCF,AlternateAlleleVCF,ReviewStatus
0,AP5Z1,Indel,Pathogenic/Likely pathogenic,1,GGAT,TGCTGTAAACTGTAACTGTAAA,"criteria provided, multiple submitters, no con..."
1,FOXRED1,single nucleotide variant,Pathogenic,1,C,T,"criteria provided, multiple submitters, no con..."
2,HFE,single nucleotide variant,Pathogenic/Likely pathogenic,1,A,C,"criteria provided, multiple submitters, no con..."
3,ABHD12,single nucleotide variant,Pathogenic,1,G,A,"criteria provided, multiple submitters, no con..."
4,HOGA1,Microsatellite,Pathogenic/Likely pathogenic,1,TGAG,T,"criteria provided, multiple submitters, no con..."
5,HOGA1,single nucleotide variant,Pathogenic,1,G,T,"criteria provided, multiple submitters, no con..."
6,HOGA1,single nucleotide variant,Pathogenic/Likely pathogenic,1,C,T,"criteria provided, multiple submitters, no con..."
7,HOGA1,single nucleotide variant,Pathogenic/Likely pathogenic,1,G,C,"criteria provided, multiple submitters, no con..."
8,HOGA1,single nucleotide variant,Pathogenic/Likely pathogenic,1,T,G,"criteria provided, multiple submitters, no con..."
9,FAM161A,single nucleotide variant,Pathogenic,1,G,A,"criteria provided, multiple submitters, no con..."


In [8]:
# Confirm saved file loads correctly
df = pd.read_csv(
    config.PROCESSED_DIR / "clinvar_filtered.tsv.gz",
    sep='\t',
    low_memory=False
)

print(f"Shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['Label'].value_counts())

print(f"\nTop 20 genes by variant count:")
print(df['GeneSymbol'].value_counts().head(20))

print(f"\nVariant type distribution:")
print(df['Type'].value_counts())

Shape: (383533, 44)

Label distribution:
Label
0    288464
1     95069
Name: count, dtype: int64

Top 20 genes by variant count:
GeneSymbol
BRCA2    6671
TTN      5835
BRCA1    4856
ATM      4486
NF1      3242
APC      3180
MSH6     2661
TSC2     2631
MSH2     2194
MLH1     1919
FBN1     1864
POLE     1725
PALB2    1642
RYR1     1639
RYR2     1552
LDLR     1533
DMD      1422
PMS2     1398
BRIP1    1388
CFTR     1370
Name: count, dtype: int64

Variant type distribution:
Type
single nucleotide variant    341826
Deletion                      23081
Duplication                   10129
Microsatellite                 5537
Indel                          1431
Insertion                      1385
Inversion                        72
Variation                        72
Name: count, dtype: int64
